# Harmony Project 3 — LoRA Fine-Tuning (Qwen2.5-7B)

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6. Then walk away — Cell 6 runs the full automated 7-run sweep.

**If a run crashes:** Restart kernel (Run → Restart) and re-run from Cell 1.

**Output:** Final adapter saved to `/kaggle/working/adapter/`. Download as zip before the session ends.

## Cell 1 — Install pinned dependencies

`transformers==4.46.3` + `trl==0.11.4` is the only combination that works with the multi-GPU LoRA flow used in Cell 6.

In [ ]:
!pip install -q \
  "transformers==4.46.3" \
  "trl==0.11.4" \
  peft bitsandbytes accelerate datasets wandb json-repair

## Cell 2 — Configuration

Only edit `BASE_MODEL_REVISION` if a newer commit on HF Hub is desired. Everything else is locked.

In [ ]:
BASE_MODEL           = "Qwen/Qwen2.5-7B-Instruct"
BASE_MODEL_REVISION  = "a09a35458c702b33eeacc393d103063234e8bc28"
WANDB_PROJECT        = "harmony-p3-lora"
SEED                 = 42

TRAIN_PATH = "/kaggle/input/datasets/keertanaks2004/harmony-ade-data/train.jsonl"
VAL_PATH   = "/kaggle/input/datasets/keertanaks2004/harmony-ade-data/val.jsonl"

print("Config loaded.")

## Cell 3 — Pre-flight checks

Verifies GPU, data files, row counts, and JSONL format. If any assert fails, **stop and fix before running Cell 6**.

In [ ]:
import os, json, torch
from pathlib import Path

# 1. GPU
assert torch.cuda.is_available(), "No GPU — this notebook requires T4×2"
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

# 2. Data files
assert Path(TRAIN_PATH).exists(), f"MISSING: {TRAIN_PATH}"
assert Path(VAL_PATH).exists(),   f"MISSING: {VAL_PATH}"

# 3. Row counts
n_train = sum(1 for _ in open(TRAIN_PATH))
n_val   = sum(1 for _ in open(VAL_PATH))
print(f"Train rows: {n_train:,}   Val rows: {n_val:,}")
assert n_train > 15000, f"train.jsonl too small ({n_train})"
assert n_val   > 2000,  f"val.jsonl too small ({n_val})"

# 4. Format spot-check
with open(TRAIN_PATH) as f:
    s = json.loads(f.readline())
assert "messages" in s and "schema_version" in s["messages"][1]["content"]
print("✓ Pre-flight passed")

## Cell 4 — Load tokenizer ONLY

**Do not load the model here.** Cell 6 loads it with a balanced GPU split. Loading it here would put 14 GB on one GPU, breaking Cell 6.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✓ Tokenizer loaded  (vocab={len(tokenizer):,})")

## Cell 5 — Load and format dataset

Applies the Qwen2.5 chat template to produce a `text` field for SFTTrainer. Token length max is ~565 after templating — `max_seq_length=600` in Cell 6 handles this with no truncation.

In [ ]:
import json
import numpy as np
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def format_example(ex):
    return {
        "text": tokenizer.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )
    }

train_dataset = Dataset.from_list(load_jsonl(TRAIN_PATH)).map(format_example)
val_dataset   = Dataset.from_list(load_jsonl(VAL_PATH)).map(format_example)
print(f"Train: {len(train_dataset):,}   Val: {len(val_dataset):,}")

# Quick token-length check (sample of 200)
lens = [len(tokenizer(ex["text"], add_special_tokens=False)["input_ids"])
        for ex in train_dataset.select(range(200))]
print(f"Token length  p50={np.percentile(lens,50):.0f}  "
      f"p95={np.percentile(lens,95):.0f}  max={max(lens)}")
assert max(lens) <= 1024, f"Some examples > 1024 tokens: {max(lens)}"
print("✓ Dataset ready")

## Cell 6 — Automated 7-run sweep (RUN AND LEAVE)

Runs **7 LoRA training jobs** sequentially:

| Phase | Runs | Settings |
|---|---|---|
| 1 — LR sweep | 3 runs | rank=16, 500 steps each, LR ∈ {1e-4, 2e-4, 5e-4} |
| 2 — Rank sweep | 3 runs | best_LR, 500 steps each, rank ∈ {8, 16, 32} |
| 3 — Final | 1 run | best_LR + best_rank, 3 epochs, **saves adapter** |

Total runtime ≈ 8–10 hours on T4×2. All metrics logged to W&B. The final adapter saves to `/kaggle/working/adapter/`.

**Add your W&B key as a Kaggle Secret named `WANDB_API_KEY`** before running this cell.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — AUTOMATED LORA SWEEP
# ════════════════════════════════════════════════════════════════════════════
import os, gc, json, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig
import wandb

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_SILENT"] = "true"

# ── Safety: free any stale model refs (in case Cell 6 was re-run) ───────────
for _v in ["model", "trainer"]:
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()
print(f"Start  →  GPU 0: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free  |  "
      f"GPU 1: {torch.cuda.mem_get_info(1)[0]/1e9:.1f} GB free")
assert torch.cuda.mem_get_info(0)[0] > 14e9, (
    "GPU 0 is not empty — restart the kernel (Run → Restart) and re-run "
    "from Cell 1. Do NOT run Cell 4's old model-load if it exists."
)
assert torch.cuda.mem_get_info(1)[0] > 14e9, (
    "GPU 1 is not empty — restart the kernel (Run → Restart) and re-run "
    "from Cell 1."
)

# ── Custom DataCollator (TRL 0.11.4 removed the built-in one) ───────────────
class DataCollatorForCompletionOnlyLM:
    def __init__(self, response_template, tokenizer):
        self.tokenizer = tokenizer
        self.response_template_ids = tokenizer.encode(
            response_template, add_special_tokens=False)

    def __call__(self, instances):
        input_ids = [torch.tensor(inst["input_ids"]) for inst in instances]
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True,
            padding_value=self.tokenizer.pad_token_id)
        labels = input_ids.clone()
        tpl = self.response_template_ids
        for i in range(len(labels)):
            seq = labels[i].tolist()
            found = False
            for j in range(len(seq) - len(tpl) + 1):
                if seq[j: j + len(tpl)] == tpl:
                    labels[i, : j + len(tpl)] = -100
                    found = True
                    break
            if not found:
                labels[i, :] = -100
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": (input_ids != self.tokenizer.pad_token_id).long(),
            "labels": labels,
        }

# ── Model loader — balanced across both T4s ─────────────────────────────────
MAX_MEM = {0: "10GiB", 1: "13GiB"}

def load_fresh_model():
    gc.collect(); torch.cuda.empty_cache()
    m = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        revision=BASE_MODEL_REVISION,
        torch_dtype=torch.float16,
        device_map="balanced_low_0",
        max_memory=MAX_MEM,
    )
    m.config.use_cache = False
    return m

# ── One training run — model ALWAYS freed via finally ───────────────────────
def run_one(run_name, learning_rate, lora_rank,
            max_steps=500, num_epochs=3, save_adapter=False):

    lora_alpha = lora_rank * 2
    out_dir    = f"/kaggle/working/checkpoints/{run_name}"
    model, trainer = None, None

    print(f"\n{'='*62}")
    print(f"  RUN : {run_name}  |  lr={learning_rate}  rank={lora_rank}")
    print(f"{'='*62}")

    try:
        model = load_fresh_model()
        print(f"  After load  →  GPU 0: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free  |  "
              f"GPU 1: {torch.cuda.mem_get_info(1)[0]/1e9:.1f} GB free")

        lora_cfg = LoraConfig(
            r=lora_rank, lora_alpha=lora_alpha,
            target_modules=["q_proj","k_proj","v_proj","o_proj",
                            "gate_proj","up_proj","down_proj"],
            lora_dropout=0.05, bias="none",
            task_type=TaskType.CAUSAL_LM,
        )

        collator = DataCollatorForCompletionOnlyLM(
            response_template="<|im_start|>assistant\n",
            tokenizer=tokenizer,
        )

        wandb.init(project=WANDB_PROJECT, name=run_name, reinit=True,
                   config={"lr": learning_rate, "rank": lora_rank,
                           "alpha": lora_alpha, "max_steps": max_steps,
                           "num_epochs": num_epochs,
                           "base_model": BASE_MODEL,
                           "revision": BASE_MODEL_REVISION})

        # Auto-resume if this run was interrupted
        resume_ckpt = None
        if Path(out_dir).exists():
            ckpts = sorted(Path(out_dir).glob("checkpoint-*"),
                           key=lambda p: int(p.name.split("-")[-1]))
            if ckpts:
                resume_ckpt = str(ckpts[-1])
                print(f"  ↩  Resuming from {resume_ckpt}")

        sft_config = SFTConfig(
            output_dir=out_dir,
            num_train_epochs=num_epochs if max_steps == -1 else 9999,
            max_steps=max_steps if max_steps != -1 else -1,
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            gradient_accumulation_steps=8,           # effective batch = 16
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            optim="adamw_torch",
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            weight_decay=0.0,
            average_tokens_across_devices=False,      # fixes multi-GPU error
            fp16=True, bf16=False,                    # T4 has no native BF16
            max_seq_length=600,                       # actual max ≈ 565
            dataset_text_field="text",
            eval_strategy="steps", eval_steps=100,
            save_strategy="steps", save_steps=100, save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss", greater_is_better=False,
            logging_steps=10,
            report_to="wandb", run_name=run_name,
            seed=SEED, data_seed=SEED,
        )

        trainer = SFTTrainer(
            model=model, args=sft_config,
            train_dataset=train_dataset, eval_dataset=val_dataset,
            data_collator=collator, peft_config=lora_cfg,
            tokenizer=tokenizer,                      # trl 0.11.4 still uses this
        )

        trainer.train(resume_from_checkpoint=resume_ckpt)
        best_eval_loss = trainer.state.best_metric
        print(f"  ✓  {run_name}  →  eval_loss={best_eval_loss:.4f}")

        if save_adapter:
            adapter_dir = "/kaggle/working/adapter"
            trainer.model.save_pretrained(adapter_dir)
            tokenizer.save_pretrained(adapter_dir)
            meta = {
                "run_name": run_name,
                "base_model": BASE_MODEL,
                "base_model_revision": BASE_MODEL_REVISION,
                "learning_rate": learning_rate,
                "lora_rank": lora_rank,
                "lora_alpha": lora_alpha,
                "best_eval_loss": round(float(best_eval_loss), 6),
                "max_seq_length": 600,
                "per_device_train_batch_size": 2,
                "gradient_accumulation_steps": 8,
                "effective_batch_size": 16,
                "optimizer": "adamw_torch",
                "fp16": True, "bf16": False,
                "target_modules": ["q_proj","k_proj","v_proj","o_proj",
                                   "gate_proj","up_proj","down_proj"],
                "lora_dropout": 0.05,
                "num_epochs": num_epochs,
            }
            with open(f"{adapter_dir}/training_args.json", "w") as f:
                json.dump(meta, f, indent=2)
            print(f"  ✓  Adapter saved → {adapter_dir}")

        wandb.finish()
        return best_eval_loss

    finally:
        # Guaranteed cleanup — runs even on exception
        if trainer is not None: del trainer
        if model   is not None: del model
        gc.collect(); torch.cuda.empty_cache()
        print(f"  Cleanup  →  GPU 0: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free  |  "
              f"GPU 1: {torch.cuda.mem_get_info(1)[0]/1e9:.1f} GB free")

# ════════════════════════════════════════════════════════════════════════════
# PHASE 1 — LR sweep (r=16 fixed)
# ════════════════════════════════════════════════════════════════════════════
lr_sweep = {
    "lora_lr_1e4_r16": 1e-4,
    "lora_lr_2e4_r16": 2e-4,
    "lora_lr_5e4_r16": 5e-4,
}
lr_results = {}
for name, lr in lr_sweep.items():
    lr_results[lr] = run_one(name, lr, lora_rank=16, max_steps=500)

best_lr = min(lr_results, key=lr_results.get)
print(f"\n✓  Best LR : {best_lr}  (eval_loss={lr_results[best_lr]:.4f})")

# ════════════════════════════════════════════════════════════════════════════
# PHASE 2 — Rank sweep (best_lr)
# ════════════════════════════════════════════════════════════════════════════
rank_sweep = {
    "lora_bestlr_r8":  8,
    "lora_bestlr_r16": 16,
    "lora_bestlr_r32": 32,
}
rank_results = {}
for name, rank in rank_sweep.items():
    rank_results[rank] = run_one(name, best_lr, lora_rank=rank, max_steps=500)

best_rank = min(rank_results, key=rank_results.get)
print(f"\n✓  Best Rank : {best_rank}  (eval_loss={rank_results[best_rank]:.4f})")

# ════════════════════════════════════════════════════════════════════════════
# PHASE 3 — Final full run (3 epochs, saves adapter)
# ════════════════════════════════════════════════════════════════════════════
final_loss = run_one(
    run_name="lora_final",
    learning_rate=best_lr,
    lora_rank=best_rank,
    max_steps=-1,
    num_epochs=3,
    save_adapter=True,
)

print(f"\n{'='*62}")
print(f"  SWEEP COMPLETE")
print(f"  best_lr   = {best_lr}")
print(f"  best_rank = {best_rank}")
print(f"  final eval_loss = {final_loss:.4f}")
print(f"  Adapter → /kaggle/working/adapter/")
print(f"{'='*62}")
print("\n⚠  Download /kaggle/working/adapter/ as a zip before the session ends.")

## Done

When Cell 6 finishes:

1. **Download** `/kaggle/working/adapter/` as a zip (right-click → Download)
2. **Note** the W&B run URLs for all 7 runs (visit https://wandb.ai/your_username/harmony-p3-lora)
3. **Commit** the adapter zip to `models/adapters/lora_v1/` in the Harmony repo

If the kernel hits the 12-hour Kaggle limit mid-sweep:
- Completed runs' checkpoints are in `/kaggle/working/checkpoints/{run_name}/`
- Re-running Cell 6 will auto-resume the interrupted run from its latest checkpoint
- But earlier completed runs' best_eval_loss is lost from memory — read it from W&B and manually skip those phases by commenting out their loops